# Exploración de las bases de datos — Corporación Favorita

Recorrido tabla por tabla: `train.csv`, `transactions.csv`, `stores.csv`, y un filtro específico de `train.csv` (tienda 44, años 2016-2017). Corre las celdas en orden.

## 0. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import os

BASE_DIR = "C:/Tesis"


## 1. train.csv

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "train.csv")

tamano_gb = os.path.getsize(FILE_PATH) / (1024**3)
print(f"Tamaño en disco: {tamano_gb:.2f} GB")


### Columnas y primeras filas
Esto solo lee un puñado de filas, no carga el archivo completo (que pesa varios GB).

In [ ]:
preview = pd.read_csv(FILE_PATH, nrows=5)
preview


In [ ]:
preview.dtypes


### Muestra aleatoria de todo el archivo
`train.csv` tiene ~125 millones de filas, así que no lo cargamos completo: lo leemos por partes (*chunks*) y de cada parte nos quedamos con un porcentaje aleatorio de filas (`SAMPLE_FRAC`). Puede tardar unos minutos — va imprimiendo el avance.

In [ ]:
SAMPLE_FRAC = 0.01     # 1% de las filas ≈ 1.2 millones -> subir/bajar según necesites
CHUNK_SIZE = 1_000_000
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)
partes = []

for i, chunk in enumerate(pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE)):
    mask = rng.random(len(chunk)) < SAMPLE_FRAC
    partes.append(chunk[mask])
    if (i + 1) % 20 == 0:
        print(f"Procesadas {(i + 1) * CHUNK_SIZE:,} filas leídas del archivo...")

train_sample = pd.concat(partes, ignore_index=True)
print(f"\nFilas en la muestra: {len(train_sample):,}")


### Resumen de columnas y tipos (sobre la muestra)

In [ ]:
train_sample.info()


### Primeras filas de la muestra

In [ ]:
train_sample.head(10)


### Nulos por columna

In [ ]:
train_sample.isna().sum()


### Valores únicos de una columna
`.unique()` te muestra cuáles son los valores distintos; `.value_counts()` además te dice cuántas veces aparece cada uno; `.nunique()` te da solo el total de valores distintos (útil cuando hay muchos, como en `item_nbr`).

In [ ]:
train_sample["onpromotion"].unique()


In [ ]:
train_sample["store_nbr"].value_counts()


In [ ]:
train_sample["item_nbr"].nunique()


### Rango de fechas y variables clave

In [ ]:
train_sample["date"] = pd.to_datetime(train_sample["date"])

print("Rango de fechas:", train_sample["date"].min(), "->", train_sample["date"].max())
print("Tiendas distintas (store_nbr):", train_sample["store_nbr"].nunique())
print("Productos distintos (item_nbr):", train_sample["item_nbr"].nunique())
print("Valores de onpromotion:", train_sample["onpromotion"].unique())


### Estadísticas de unit_sales (la variable a predecir)

In [ ]:
train_sample["unit_sales"].describe()


---
## 2. transactions.csv
Esta tabla es chica (una fila por tienda y día), no necesita muestreo — se carga completa.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "transactions.csv")
transactions = pd.read_csv(FILE_PATH)
transactions.head()


In [ ]:
transactions.info()


### Tienda con más transacciones
Dos formas de mirarlo: el **total acumulado** por tienda (sumando todos los días), o el **pico** más alto en un solo día.

In [ ]:
# Total acumulado por tienda (de mayor a menor)
transactions.groupby("store_nbr")["transactions"].sum().sort_values(ascending=False)


In [ ]:
tienda_top = transactions.groupby("store_nbr")["transactions"].sum().idxmax()
print("Tienda con más transacciones en total:", tienda_top)


In [ ]:
# Día con más transacciones en una sola tienda (el pico más alto)
transactions.loc[transactions["transactions"].idxmax()]


---
## 3. stores.csv
54 tiendas en total — tabla chica, se carga completa.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "stores.csv")
stores = pd.read_csv(FILE_PATH)
stores.head()


### Info de la tienda 44

In [ ]:
stores[stores["store_nbr"] == 44].squeeze()


---
## 4. train.csv filtrado — Tienda 44, años 2016 y 2017
A diferencia de la sección 1 (que usaba una muestra aleatoria), acá queremos **todas** las filas de la tienda 44 en 2016-2017, sin sub-muestrear. Como solo es una tienda, el resultado final es chico aunque el archivo de entrada sea gigante — igual lo leemos por partes (*chunks*) para no cargar los 125 millones de filas en memoria de una vez.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "train.csv")
CHUNK_SIZE = 1_000_000
STORE_NBR = 44   # en train.csv viene como número, no como texto

partes = []

for i, chunk in enumerate(pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE)):
    # Filtro rápido por año usando el texto de la fecha (más rápido que parsear todo a datetime)
    filtro_anio = chunk["date"].str[:4].isin(["2016", "2017"])
    filtro_tienda = chunk["store_nbr"] == STORE_NBR

    partes.append(chunk[filtro_anio & filtro_tienda])

    if (i + 1) % 20 == 0:
        print(f"Procesadas {(i + 1) * CHUNK_SIZE:,} filas leídas del archivo...")

train_tienda44 = pd.concat(partes, ignore_index=True)
train_tienda44["date"] = pd.to_datetime(train_tienda44["date"])

print(f"\nFilas obtenidas: {len(train_tienda44):,}")
train_tienda44.head()


---
## Pendiente
Secciones para `test.csv`, `items.csv`, `oil.csv` y `holidays_events.csv` — todas son tablas chicas (no necesitan muestreo): `pd.read_csv(FILE_PATH)` directo y después `.info()`, `.head()`, `.isna().sum()`, `.describe()` igual que en `stores.csv` / `transactions.csv`.